# Pipeline Landing para Bronze — CineData Analytics

Este notebook realiza a ingestão dos dados brutos da camada **Landing** (arquivos CSV e API do Banco Central) para a camada **Bronze** em formato Parquet.

### Padrões Técnicos Aplicados:
- **Esquemas Canônicos e Validação**: Definição de `StructType` com validação de conformidade de colunas para cada fonte de dados.
- **Rastreabilidade**: Adição do carimbo de data e hora (`ingestion_datetime`) em todas as tabelas persistidas na Bronze.

### Premissa de Arquitetura e Duplicação de código dos Notebooks:
- Funções utilitárias (como persistência, validação e qualidade) foram mantidas diretamente em cada notebook. Essa duplicação visa dispensar módulos compartilhados ou abstrações externas no Databricks.


In [1]:
from datetime import datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import requests
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# Resolução de diretórios da Landing Zone no Volume gerenciado do Unity Catalog
LANDING_DIR = Path("/Volumes/workspace/default/cinedata/landing")

# Obtenção da Sessão Spark gerenciada no Databricks Serverless
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", "false")

# Provisionamento do schema/database da camada Bronze no catálogo gerenciado
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

def write_dataframe_with_timestamp(
    dataframe: DataFrame,
    table_name: str,
    save_mode: str = "append"
) -> None:
    """Grava o DataFrame em formato Delta no catálogo, adicionando timestamp de ingestão."""
    dataframe_with_timestamp = dataframe.withColumn("ingestion_datetime", current_timestamp())
    (
        dataframe_with_timestamp.write
        .format("delta")
        .mode(save_mode)
        .saveAsTable(table_name)
    )

def validate_landing_zone_artifacts(
    landing_path: Path,
    expected_filenames: list[str]
) -> bool:
    """Verifica a presença e o tamanho dos arquivos CSV esperados na Landing Zone."""
    existing_files = {
        file_entry.name: file_entry.stat().st_size
        for file_entry in landing_path.glob("*.csv")
    }
    missing_files = [filename for filename in expected_filenames if filename not in existing_files]

    if missing_files:
        print(f"[ALERTA CRÍTICO] Arquivos ausentes na Landing Zone ({landing_path}):")
        for missing_filename in missing_files:
            print(f"  - ❌ {missing_filename}")
        return False

    print(f"[SUCESSO] Todos os {len(expected_filenames)} arquivos de entrada estão disponíveis:")
    for filename in expected_filenames:
        file_size_kb = existing_files[filename] / 1024
        print(f"  - ✅ {filename} ({file_size_kb:.1f} KB)")
    return True


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/20 21:51:32 WARN Utils: Your hostname, DESKTOP-QSIGGPV, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/20 21:51:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 21:51:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/miguelsb/workspace/visagio/.venv/lib/python3.14/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
RuntimeError: ree

Py4JError: An error occurred while calling None.org.apache.spark.sql.classic.SparkSession

## 1. Ingestão de Arquivos CSV da Camada Landing

Leitura e conformação de esquemas para cada conjunto de dados brutos antes da persistência em formato Parquet na camada Bronze.
* A coluna `tconst` está no dataset `movies_info_TMDB_IMDB` não foi considerada por não aparecer no contrato

In [ ]:
LANDING_INGESTION_SPECS = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews",
}


# Checagem de presença física e volumetria dos arquivos da Landing Zone
existing_files = {
    file_entry.name: file_entry.stat().st_size
    for file_entry in LANDING_DIR.glob("*.csv")
}
missing_files = [filename for filename in LANDING_INGESTION_SPECS if filename not in existing_files]

if missing_files:
    print(f"[ALERTA] Arquivos ausentes na Landing Zone ({LANDING_DIR}):")
    for missing_filename in missing_files:
        print(f"  - ❌ {missing_filename}")
else:
    print(f"[SUCESSO] Todos os {len(LANDING_INGESTION_SPECS)} arquivos obrigatórios estão disponíveis:")
    for filename in LANDING_INGESTION_SPECS:
        file_size_kb = existing_files[filename] / 1024
        print(f"  - ✅ {filename} ({file_size_kb:.1f} KB)")


In [ ]:
LANDING_INGESTION_SPECS = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews",
}

# Ingestão dos arquivos brutos CSV para a camada Bronze em formato Delta (modo append),
# preservando os dados no estado original (todas as colunas como string)
# e carimbando a coluna de rastreabilidade temporal ingestion_datetime.
for source_filename, bronze_table_name in LANDING_INGESTION_SPECS.items():
    raw_dataframe = spark.read.csv(str(LANDING_DIR / source_filename), header=True, inferSchema=False)
    write_dataframe_with_timestamp(
        dataframe=raw_dataframe,
        table_name=bronze_table_name,
        save_mode="append"
    )
    print(f"Ingestão concluída com sucesso: {bronze_table_name}")


Ingestão concluída com sucesso: bronze.tb_movies_info


Ingestão concluída com sucesso: bronze.tb_movies_financials


Ingestão concluída com sucesso: bronze.tb_movies_metrics


Ingestão concluída com sucesso: bronze.tb_credits_and_tags


Ingestão concluída com sucesso: bronze.tb_movies_reviews


## 2. Ingestão da API do Banco Central (Cotação do Dólar)

Consulta à API PTAX Olinda do BACEN para obtenção do histórico de cotações de compra do Dólar americano.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class CotacaoItem(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    
    cotacao_compra: float = Field(alias="cotacaoCompra")
    data_hora_cotacao: str = Field(alias="dataHoraCotacao")

class PtaxResponse(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    odata_context: str = Field(alias="@odata.context")
    value: list[CotacaoItem]


In [ ]:
# Formato esperado de data pela API PTAX Olinda: MM-DD-AAAA.
# Suporte a parâmetros (widgets) no Databricks com fallback automatizado para os últimos 7 dias.
BACEN_API_DATE_FORMAT = "%m-%d-%Y"
QUERY_INTERVAL_DAYS = 7
RECIFE_TIMEZONE = "America/Recife"
DOLAR_QUOTE_ENDPOINT = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    "@dataInicial='{start_date}'&@dataFinalCotacao='{end_date}'&=dataHoraCotacao,cotacaoCompra&=json"
)
BacenCotacaoBronzeSchema = StructType([
    StructField("cotacaoCompra", DoubleType(), nullable=True),
    StructField("dataHoraCotacao", StringType(), nullable=True),
])

current_datetime_recife = datetime.now(ZoneInfo(RECIFE_TIMEZONE))
calculated_default_end = current_datetime_recife.strftime(BACEN_API_DATE_FORMAT)
calculated_default_start = (current_datetime_recife - timedelta(days=QUERY_INTERVAL_DAYS)).strftime(BACEN_API_DATE_FORMAT)

dbutils.widgets.text("data_inicio", calculated_default_start, "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", calculated_default_end, "Data Fim (MM-DD-AAAA)")
start_date_param = dbutils.widgets.get("data_inicio") or calculated_default_start
end_date_param = dbutils.widgets.get("data_fim") or calculated_default_end

response = requests.get(
    DOLAR_QUOTE_ENDPOINT.format(
        start_date=start_date_param,
        end_date=end_date_param
    )
)
response.raise_for_status()
validated_response = PtaxResponse.model_validate(response.json())
cotacao_records = [cotacao_item.model_dump(by_alias=True) for cotacao_item in validated_response.value]

dataframe_cotacao = spark.createDataFrame(data=cotacao_records, schema=BacenCotacaoBronzeSchema)
write_dataframe_with_timestamp(
    dataframe=dataframe_cotacao,
    table_name="bronze.tb_cotacao_dolar",
    save_mode="append"
)
display(dataframe_cotacao)


DataFrame[cotacaoCompra: double, dataHoraCotacao: string]